<div align="center">

# Week 01: Python and cardiology data basics

**Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists**<br>
Turkish Society of Cardiology · Prof. Dr. Utku Kose

[Week page](README.md) · [Lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html) · [Interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lab.html) · [PDF notes](Week01_Lecture_Notes.pdf) · [Open in Colab (fallback)](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/NB01_python_and_ecg_basics.ipynb)

</div>

**How to use this Jupyter notebook.** Run the cells from top to bottom with Shift+Enter. Section 0 prepares the environment and must run first. Exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells: Open them only after trying. *Observe and reflect* cells ask open questions and have no solution cell. *Python note* cells explain the language where a new construct first appears. A participant who opens the notebook in the middle runs the *Catch-up* cell at the start of that section.

**Three paths.** All cells without a label form the **core path**, which is enough for everybody, including those who have never written code. Cells labelled **Advanced path** are optional extensions for those who already program. The **prompt path** reaches the same results by writing prompts instead of code. Every exercise has a collapsed *Prompt route* with a ready prompt for an artificial intelligence (AI) assistant, and every section ends with a *Prompt challenge*. Copy the prompt into an assistant, replace the content of the exercise cell with the code it returns and run it, and let the check decide. The [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md) of the week collects all prompts, and the [guide to the prompt path](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/PROMPT_PATH.md) explains which assistants work with this notebook.

**Where it runs.** The notebook is written for Jupyter in Anaconda on your own computer (see SETUP.md in the course folder). If installing Anaconda is not possible, the same notebook opens in Google Colab through the link at the top. Every data set here is synthetic or comes from a public research database. Real patient data belongs only on your own computer, under the rules of your institution, and never in a cloud notebook or in an artificial intelligence (AI) chat.

## 0. Setup

The first cell finds the course folder. If the notebook was opened on its own, for example in Google Colab, it downloads the course package `vclinic` and the data files from GitHub once. The second cell loads the packages, chooses the language of the messages and checks the optional packages of the week.

In [ ]:
#@title Course files: run this cell (there is no need to open it)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
    "data/uci-heart-disease/processed.cleveland.data",
    "data/physionet/mitdb/100.hea",
    "data/physionet/mitdb/100.dat",
    "data/physionet/mitdb/100.atr",
    "data/physionet/ptbdb/patient001/s0010_re.hea",
    "data/physionet/ptbdb/patient001/s0010_re.dat",
    "data/physionet/ptbdb/patient001/s0010_re.xyz",
    "data/examples/example_caliper_measurements.csv",
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # A downloaded copy, for example in Colab: the files of the package are refreshed.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Course files downloaded to", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Setup: packages, language and plotting style
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.set_language("en")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026

print("vclinic version", vclinic.__version__)
print("Course folder:", ROOT)
for name in ["wfdb", "neurokit2", "ipywidgets"]:
    try:
        importlib.import_module(name)
        print(f"{name}: installed")
    except ImportError:
        print(f"{name}: not installed (optional; the notebook uses a built-in fallback)")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
teach.register_answers({
    "Exercise 1.1": (0.833, 0.001),
    "Exercise 1.2": (439.1, 0.2),
    "Exercise 1.3": (2, 0),
    "Exercise A": (56.6, 0.1),
    "Exercise B": (32.0, 0.1),
    "Exercise C": (47, 0),
    "Exercise D": (100, 0),
    "Exercise E": (0.104, 0.002),
    "Exercise F": (403.9, 0.3),
    "Exercise G": (216, 6),
    "Exercise H": (33, 0),
    "Exercise I": (39.05, 0.1),
    "Exercise J": (466.5, 0.2),
})
print("Expected answers are ready.")

## Python step 1: Values, lists, dictionaries, decisions, loops and functions

*Lecture: [Python step 1: The language of the clinic](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#python-step)*

Python is the language of this course. A code cell holds Python instructions, and Shift+Enter runs them. Text after `#` is a comment: Python skips it, and it explains the code to the reader. Every cell of this step begins with the line `# Python practice: a small example for learning the language`. The examples use the numbers of the case of the week, a 67-year-old woman with a heart rate of 85 beats per minute (bpm) and a QT interval of 391 ms.

In [ ]:
# Python practice: a small example for learning the language
heart_rate = 85          # beats per minute (bpm)
qt_ms = 391              # QT interval in milliseconds
rr_s = 60 / heart_rate   # RR interval in seconds
print(rr_s)
print(round(rr_s, 3))

**Python note.** `heart_rate = 85` creates a *variable*: a name that holds a value. The sign `=` assigns; it does not test equality. `/` divides, and `60 / heart_rate` gives the RR interval in seconds. `print()` shows a value, and `round(value, 3)` rounds it to three decimals. Python counts in floating-point numbers, so `0.7058823529411765` is printed in full until it is rounded.

In [ ]:
# Python practice: a small example for learning the language
patient_id = "VC-0317"
print(f"Patient {patient_id}: {heart_rate} bpm, RR {rr_s:.3f} s")

**Python note.** Text in quotes is a *string*. A string that starts with `f` is an *f-string*: Values inside curly braces are put into the text, and `{rr_s:.3f}` writes the number with three decimals.

In [ ]:
# Python practice: a small example for learning the language
troponin = [3.6, 4.1, 3.9]          # serial measurements (ng/L)
print(troponin[0], troponin[-1], len(troponin))
troponin.append(4.0)                # a fourth measurement arrives
print(troponin, "max:", max(troponin))

**Python note.** Square brackets make a *list*, an ordered series of values. Python counts positions from 0, so `troponin[0]` is the first value and `troponin[-1]` the last. `len()` counts the elements, `.append()` adds one at the end, and `max()` returns the largest.

In [ ]:
# Python practice: a small example for learning the language
patient = {"id": "VC-0317", "age": 67, "sex": "F", "hr_bpm": 85, "qt_ms": 391}
print(patient["age"])
patient["lvef"] = 63                # a new field: left ventricular ejection fraction (%)
print(patient)

**Python note.** Curly braces with `key: value` pairs make a *dictionary*, which works like a patient chart: Each field has a label. `patient["age"]` reads a field, and `patient["lvef"] = 63` adds one.

In [ ]:
# Python practice: a small example for learning the language
qtc_ms = 465
limit = 460 if patient["sex"] == "F" else 450
if qtc_ms >= limit:
    print(f"QTc {qtc_ms} ms is at or above the limit of {limit} ms")
else:
    print(f"QTc {qtc_ms} ms is below the limit of {limit} ms")

**Python note.** `if` runs the indented lines below it only when its condition is true, and `else` covers the other case. The indentation (four spaces) marks which lines belong to each branch. `==` tests equality and `>=` means "greater than or equal". The line `limit = 460 if ... else 450` chooses one of two values in a single line. The two limits are the thresholds of a prolonged QTc for women and men in the 2009 recommendations of the American Heart Association (AHA) and its partners.

In [ ]:
# Python practice: a small example for learning the language
clinic = [
    {"id": "VC-0317", "hr": 85, "qt": 391},
    {"id": "VC-0079", "hr": 58, "qt": 404},
    {"id": "VC-0254", "hr": 98, "qt": 343},
]
for p in clinic:
    rr = 60 / p["hr"]
    print(p["id"], round(rr, 3))

**Python note.** `clinic` is a list of dictionaries, one per patient. `for p in clinic:` takes the patients one by one, and the indented lines run once for each of them, like a ward round.

In [ ]:
# Python practice: a small example for learning the language
def qtc_bazett(qt_ms, rr_s):
    """QT interval corrected for heart rate with Bazett's formula (ms)."""
    return qt_ms / rr_s ** 0.5


print(round(qtc_bazett(391, 60 / 85), 1))

**Python note.** `def` defines a *function*, a named procedure that can be used again, much like a clinical protocol. `qt_ms` and `rr_s` are its inputs, and `return` gives back the result. The text in triple quotes is the docstring, the short description of the function. `**` raises to a power, so `rr_s ** 0.5` is the square root.

In [ ]:
# Python practice: a small example for learning the language
rr = np.array([0.81, 0.85, 0.83])   # three RR intervals (s)
print(60 / rr)                      # heart rate of each beat (bpm)
print(rr.mean(), rr.std(ddof=1))

**Python note.** NumPy, imported in Section 0 as `np`, works on whole arrays of numbers at once: `60 / rr` divides 60 by each RR interval. `.mean()` returns the mean, and `.std(ddof=1)` the sample standard deviation, which divides by $n-1$.

**Quick check.** After `troponin.append(4.0)`, what does `troponin[-1]` return?

<details><summary>Answer</summary><code>4.0</code>: the index <code>-1</code> always returns the last element, and <code>append()</code> put <code>4.0</code> at the end.</details>

**Exercise 1.1.** Compute the RR interval in seconds for a heart rate of 72 bpm and round it to three decimals.

<details><summary><b>Prompt route</b> for Exercise 1.1: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: A heart rate of 72 beats per minute (bpm).
Task: Compute the RR interval in seconds with RR = 60 / HR.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: Round the RR interval to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise 1.1", answer)`. Return only the code in one block, with a short comment above each step.
Test: Print also the RR interval at 60 bpm, which must be exactly 1.0 s.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise 1.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(60 / 72, 3)
teach.check("Exercise 1.1", answer)

**Exercise 1.2.** Write a function `qtc_fridericia(qt_ms, rr_s)` that divides the QT interval by the cube root of the RR interval, $QT_{cF} = QT / RR^{1/3}$. Use it for the case of the week (QT 391 ms, heart rate 85 bpm) and round to one decimal.

<details><summary><b>Prompt route</b> for Exercise 1.2: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: The QT interval in milliseconds and the RR interval in seconds. The case of the week has a QT of 391 ms at a heart rate of 85 bpm, so RR = 60 / 85 s.
Task: Write a function `qtc_fridericia(qt_ms, rr_s)` that returns the QT interval corrected with Fridericia's formula, QTc = QT / RR^(1/3). Use it for the case of the week.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: Round the QTc of the case to one decimal. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise 1.2", answer)`. Return only the code in one block, with a short comment above each step.
Test: Check with assert statements that qtc_fridericia(400, 1.0) returns 400.0 and that qtc_fridericia(400, 0.729) returns 444.4 after rounding, because 0.729 is 0.9 cubed.
</pre>

</details>

In [ ]:
def qtc_fridericia(qt_ms, rr_s):
    return None   # replace None with the formula


answer = qtc_fridericia(391, 60 / 85)
answer = round(answer, 1) if answer is not None else None
teach.check("Exercise 1.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def qtc_fridericia(qt_ms, rr_s):
    """QT interval corrected with Fridericia's formula (ms)."""
    return qt_ms / rr_s ** (1 / 3)


answer = round(qtc_fridericia(391, 60 / 85), 1)
teach.check("Exercise 1.2", answer)

**Exercise 1.3.** Count the patients of the list `clinic` whose heart rate is above 80 bpm. Use a `for` loop and an `if`.

<details><summary><b>Prompt route</b> for Exercise 1.3: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: A list named `clinic` holds patients as dictionaries with the keys "id", "hr" (heart rate in bpm) and "qt" (QT interval in ms), for example {"id": "VC-0317", "hr": 85, "qt": 391}.
Task: Count the patients whose heart rate is above 80 bpm.
Constraints: Use a for loop and an if statement, without pandas. Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The number of patients. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise 1.3", answer)`. Return only the code in one block, with a short comment above each step.
Test: For a list with the heart rates 72, 85 and 81, the same loop must count 2.
</pre>

</details>

In [ ]:
count = 0
# your loop here
answer = None   # your answer
teach.check("Exercise 1.3", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
count = 0
for p in clinic:
    if p["hr"] > 80:
        count = count + 1
answer = count
teach.check("Exercise 1.3", answer)

## 1. The patient registry: The UCI Heart Disease data with pandas

*Lecture: [Patient tables with pandas](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#patient-tables-with-pandas)*

The Cleveland data of the UCI Heart Disease collection hold 14 variables of 303 patients who underwent coronary angiography. The column `num` grades the disease from 0 (no narrowing of more than 50 percent) to 4. This section is the in-class mini project of the week: Load the data, summarise them and draw the first figures.

**From the formulas to the code.** The mean $\bar{x}$ is `df["chol"].mean()`. The sample standard deviation $s$, with $n-1$ in the denominator, is `df["chol"].std()`. The median is `.median()`, and the interquartile range is `.quantile(0.75) - .quantile(0.25)`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 1 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(1)

In [ ]:
from vclinic.datasets import load_cleveland

heart = load_cleveland()
print(heart.shape)          # (rows, columns)
heart.head()

**Python note.** pandas, imported as `pd`, keeps a table in a *DataFrame*: rows are patients and columns are variables. `load_cleveland()` reads the bundled file with `pd.read_csv` and names the columns. `.shape` gives the number of rows and columns, and `.head()` shows the first five rows.

| Column | Meaning |
|---|---|
| `age`, `sex` | Age in years; sex (1 = male, 0 = female) |
| `cp` | Chest pain type (1 typical angina, 2 atypical angina, 3 non-anginal pain, 4 asymptomatic) |
| `trestbps` | Resting systolic blood pressure (mmHg) |
| `chol` | Serum cholesterol (mg/dL) |
| `fbs` | Fasting blood sugar above 120 mg/dL (1 = yes) |
| `restecg` | Resting ECG (0 normal, 1 ST-T abnormality, 2 left ventricular hypertrophy) |
| `thalach` | Maximum heart rate reached in the exercise test (bpm) |
| `exang` | Exercise-induced angina (1 = yes) |
| `oldpeak` | ST depression induced by exercise relative to rest (mm) |
| `slope` | Slope of the peak exercise ST segment (1 up, 2 flat, 3 down) |
| `ca` | Number of major vessels coloured by fluoroscopy (0 to 3) |
| `thal` | Thallium scan (3 normal, 6 fixed defect, 7 reversible defect) |
| `num`, `disease` | Angiographic disease grade 0 to 4; `disease` = 1 when `num` > 0 |

In [ ]:
# The worked example of the lecture: five cholesterol values
chol5 = heart["chol"].head(5)
print(list(chol5))
mean = chol5.sum() / len(chol5)
sd = (((chol5 - mean) ** 2).sum() / (len(chol5) - 1)) ** 0.5
print(f"mean by hand: {mean:.2f}  pandas: {chol5.mean():.2f}")
print(f"SD by hand: {sd:.2f}  pandas: {chol5.std():.2f}")
print(f"median: {chol5.median():.1f}")

In [ ]:
# Summary of the main variables, by disease status
cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
print(heart["disease"].value_counts())
heart.groupby("disease")[cols].agg(["mean", "std"]).round(1)

**Python note.** `value_counts()` counts each value of a column. `groupby("disease")` splits the table into the patients with and without disease, and `.agg(["mean", "std"])` computes the two statistics for each group and each column.

In [ ]:
# Two first figures of the registry
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for value, color, label in [(0, "#1F5F8B", "no disease"),
                            (1, "#9B1B1B", "disease")]:
    sub = heart[heart["disease"] == value]
    axes[0].hist(sub["age"], bins=15, alpha=0.6, color=color, label=label)
axes[0].set_xlabel("Age (years)")
axes[0].set_ylabel("Number of patients")
axes[0].legend()
heart.boxplot(column="thalach", by="disease", ax=axes[1])
axes[1].set_title("")
axes[1].set_xlabel("Disease (0 no, 1 yes)")
axes[1].set_ylabel("Maximum heart rate (bpm)")
fig.suptitle("")
plt.tight_layout()
plt.show()

**Exercise A.** What is the mean age of the patients with disease (`disease == 1`)? Round to one decimal.

<details><summary><b>Prompt route</b> for Exercise A: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: A pandas DataFrame named `heart` holds the Cleveland data of the UCI Heart Disease data set, one row per patient. The column age is in years, and the column disease is 1 for coronary artery disease and 0 for none.
Task: Compute the mean age of the patients with disease equal to 1.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: Round the mean age to one decimal. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise A", answer)`. Return only the code in one block, with a short comment above each step.
Test: Print the number of patients in each disease group; the two numbers must add up to 303.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(heart.loc[heart["disease"] == 1, "age"].mean(), 1)
teach.check("Exercise A", answer)

**Exercise B.** What percentage of the 303 patients are women (`sex == 0`)? Round to one decimal.

<details><summary><b>Prompt route</b> for Exercise B: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: The DataFrame `heart`, with the column sex coded 1 for men and 0 for women.
Task: Compute the percentage of women among all patients.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: Round the percentage to one decimal. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise B", answer)`. Return only the code in one block, with a short comment above each step.
Test: Compute the percentage of men in the same way and print the sum of the two percentages, which must be 100.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise B", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(100 * (heart["sex"] == 0).mean(), 1)
teach.check("Exercise B", answer)

**Observe and reflect.** Which variable differs most between the two groups in the summary table? Is the direction of the difference clinically plausible? What does the small number of women tell about how far the data can be generalised?

**Advanced path.** Two columns hold missing values. Count them with `heart.isna().sum()`, then compare the disease rate by chest pain type with `pd.crosstab(heart["cp"], heart["disease"], normalize="index")`.

In [ ]:
# Advanced path
print(heart.isna().sum()[heart.isna().sum() > 0])
pd.crosstab(heart["cp"], heart["disease"], normalize="index").round(2)

**Prompt challenge, Section 1: Cholesterol by disease status.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: The DataFrame `heart` with the columns chol, the serum cholesterol in mg/dL, and disease, 1 for coronary artery disease and 0 for none.
Task: Draw box plots of chol for the two disease groups side by side, with axis labels and units, and print the median cholesterol of each group.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure and the two medians. Return only the code in one block, with a short comment above each step.
Test: Print the size of each group; the two sizes must add up to 303.
</pre>

</details>

**What to check.** The medians are 234.5 mg/dL without disease and 249.0 mg/dL with disease.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 2. The virtual clinic opens

*Lecture: [The virtual cardiology clinic, the project of the course](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#the-virtual-cardiology-clinic-the-project-of-the-course)*

`vclinic.make_cohort()` creates the registry of the virtual cardiology clinic. Each patient has demographic data, risk factors, laboratory values, a left ventricular ejection fraction (LVEF), a main diagnosis and the parameters of a resting 12-lead electrocardiogram (ECG). A patient keeps the same values in every week of the course. The data are synthetic: They follow broad clinical patterns but describe no real person.

**Started here?** Run the catch-up cell below once. It runs the code that Section 2 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(2)

In [ ]:
clinic = vclinic.make_cohort(n=500, seed=2026)
print(clinic.shape)
clinic.head()

**Python note.** `vclinic.make_cohort(n=500, seed=2026)` calls a function of the course package with two *keyword arguments*. The seed fixes the random numbers, so every run gives the same 500 patients.

In [ ]:
# Main diagnoses and rhythm of the clinic
print(clinic["main_diagnosis"].value_counts())
print(clinic["ecg_rhythm"].value_counts())
clinic[["age", "bmi", "sbp", "ldl", "hdl", "egfr", "lvef", "ntprobnp",
        "ecg_hr", "ecg_pr", "ecg_qrs", "ecg_qt"]].describe().round(1)

The diagnosis codes are the international abbreviations: `CCS` chronic coronary syndrome, `ACS` acute coronary syndrome, `HFrEF` heart failure with reduced ejection fraction, `HFpEF` heart failure with preserved ejection fraction and `none` no structural heart disease. `AF` in the rhythm column is atrial fibrillation. The full data dictionary is in DATASETS.md.

In [ ]:
# The virtual clinic next to the Cleveland data
compare = pd.DataFrame({
    "Cleveland (real)": [
        heart["age"].mean(), 100 * heart["sex"].mean(),
        heart["trestbps"].mean(), heart["chol"].mean(),
        100 * heart["disease"].mean()],
    "Virtual clinic (synthetic)": [
        clinic["age"].mean(), 100 * (clinic["sex"] == "M").mean(),
        clinic["sbp"].mean(), clinic["total_chol"].mean(),
        100 * clinic["cad"].mean()],
}, index=["Mean age (years)", "Men (%)",
          "Systolic pressure (mmHg)",
          "Total cholesterol (mg/dL)",
          "Coronary disease (%)"])
compare.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(heart["age"], bins=20, density=True, alpha=0.55, color="#1F5F8B",
        label="Cleveland (real)")
ax.hist(clinic["age"], bins=20, density=True, alpha=0.55, color="#9B1B1B",
        label="Virtual clinic (synthetic)")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

**Exercise C.** How many patients of the virtual clinic have the main diagnosis `HFrEF`?

<details><summary><b>Prompt route</b> for Exercise C: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: A pandas DataFrame named `clinic` holds the 500 synthetic patients of the virtual clinic. Its column main_diagnosis holds one of the labels "none", "CCS", "HFpEF", "HFrEF" and "ACS".
Task: Count the patients whose main diagnosis is exactly "HFrEF".
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The number of patients as an integer. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise C", answer)`. Return only the code in one block, with a short comment above each step.
Test: Print the counts of all diagnoses with value_counts(); they must add up to 500.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise C", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int((clinic["main_diagnosis"] == "HFrEF").sum())
teach.check("Exercise C", answer)

**Observe and reflect.** The two cohorts look alike in the table. In which ways could a synthetic registry still be wrong in a way that matters for a model trained on it? Think of rare conditions, of correlations between variables and of measurement noise.

**Prompt challenge, Section 2: Diagnoses by sex.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: The DataFrame `clinic` of the virtual clinic with the columns main_diagnosis and sex, coded "F" or "M".
Task: Make a table of the number of patients for each main diagnosis by sex with pd.crosstab, with row and column totals.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The table. Return only the code in one block, with a short comment above each step.
Test: The grand total must be 500.
</pre>

</details>

**What to check.** The clinic holds 221 women and 279 men. Of the 47 patients with HFrEF, 16 are women and 31 are men.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 3. The ECG as data: Samples, sampling frequency and units

*Lecture: [The ECG as data](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#the-ecg-as-data)*

**From the formulas to the code.** The sampling frequency $f_s$ is `case.fs`. The time of sample $n$, $t_n = n / f_s$, is in `case.time`. The number of samples in an interval, $N = \Delta t \cdot f_s$, is `round(dt * case.fs)`. The voltages in millivolts are the columns of `case.leads`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 3 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(3)

In [ ]:
case = vclinic.patient_ecg(clinic, "VC-0317")   # the case of the week
print(case)
print("samples per lead:", len(case.leads))
print("time step (s):", 1 / case.fs)
case.leads.head()

In [ ]:
# Lead II on ECG paper: 10 seconds and one beat
fig, axes = plt.subplots(2, 1, figsize=(12, 6.5))
teach.plot_strip(axes[0], case.time, case.lead("II"), label="II")
one = (case.time >= 1.0) & (case.time <= 2.0)
teach.plot_strip(axes[1], case.time[one], case.lead("II")[one], label="II")
axes[1].plot(case.time[one][::10], case.lead("II")[one][::10], "o",
             color="#9B1B1B", ms=3, label="every 10th sample")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.show()

**Python note.** `case.time >= 1.0` compares every time value with 1.0 and returns an array of `True` and `False`, a *mask*. Two masks joined with `&` keep the samples between 1 and 2 seconds, and `x[mask]` selects them. `[::10]` takes every tenth element.

In [ ]:
# Samples in one RR interval at three sampling frequencies
rr = 60 / 85
for fs in [500, 360, 1000]:
    print(f"fs = {fs} Hz: RR = {rr:.3f} s spans {round(rr * fs)} samples")

**Explore.** The panel samples one heartbeat of the case again at a lower frequency and stores it with fewer bits, as an analog-to-digital converter (ADC) would do. The reference is the same beat at 2000 Hz. Move the sliders and watch how the height and the time of the R peak change. Below about 100 Hz the R peak is clipped and shifted. With few bits the trace turns into steps.

In [ ]:
# Explore: sampling frequency and resolution
fine = vclinic.simulate(case.params, duration_s=2.0, fs=2000, noise="none", seed=7)
t_ref, x_ref = fine.time, fine.lead("II")


def show_sampling(fs_hz=500, bits=12):
    step = int(round(2000 / fs_hz))
    t_s, x_s = t_ref[::step], x_ref[::step]
    q = 10.0 / 2 ** bits                       # step size over a 10 mV range
    x_q = np.round(x_s / q) * q
    window = (t_ref > 0.25) & (t_ref < 0.75)
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(t_ref[window], x_ref[window], color="#888888", lw=1,
            label="2000 Hz reference")
    w2 = (t_s > 0.25) & (t_s < 0.75)
    ax.step(t_s[w2], x_q[w2], where="mid", color="#9B1B1B", lw=1.2,
            label=f"{fs_hz} Hz, {bits} bit")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("mV")
    ax.legend(loc="upper right")
    plt.show()
    print(f"R peak: reference {x_ref.max():.3f} mV, "
          f"sampled {x_q.max():.3f} mV; "
          f"time shift {abs(t_s[np.argmax(x_q)] - t_ref[np.argmax(x_ref)]) * 1000:.1f} ms; "
          f"resolution {q * 1000:.2f} µV")


try:
    from ipywidgets import interact, IntSlider
    interact(show_sampling,
             fs_hz=IntSlider(value=500, min=50, max=1000, step=50, description="fs (Hz)"),
             bits=IntSlider(value=12, min=4, max=16, step=1, description="bits"))
except ImportError:
    show_sampling(500, 12)
    show_sampling(100, 6)

**Exercise D.** On ECG paper one large box lasts 0.2 s. How many samples does one large box hold at 500 Hz?

<details><summary><b>Prompt route</b> for Exercise D: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: An ECG sampled at 500 Hz. One large box of ECG paper lasts 0.2 s.
Task: Compute how many samples one large box holds, N = duration × sampling frequency.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The number of samples as an integer. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise D", answer)`. Return only the code in one block, with a short comment above each step.
Test: One small box of 0.04 s must hold 20 samples at the same sampling frequency.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise D", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(0.2 * 500)
teach.check("Exercise D", answer)

**Prompt challenge, Section 3: One second of lead II, sample by sample.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: `case` is the ECG of the case of the week: case.time holds the times in seconds, case.lead("II") the voltages of lead II in millivolts and case.fs the sampling frequency in Hz.
Task: Plot the first second of lead II with a small marker at every sample, with the time axis in milliseconds and the voltage axis in millivolts, and print the number of plotted samples.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure and one number. Return only the code in one block, with a short comment above each step.
Test: At 500 Hz one second holds 500 samples.
</pre>

</details>

**What to check.** The printed number is 500, or 501 if both ends were included, and the time axis runs from 0 to 1000 ms.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 4. Twelve leads from one heart vector

*Lecture: [The heart as a vector: Twelve leads, one source](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#the-heart-as-a-vector-twelve-leads-one-source)*

**From the formulas to the code.** The heart vector $\mathbf{H} = (H_x, H_y, H_z)$ is a NumPy array of three numbers. The lead vectors $\mathbf{L}$ of the 12 leads are the rows of `vclinic.LEAD_MATRIX`. The lead voltage $V_L = \mathbf{L} \cdot \mathbf{H}$ for all leads at once is `vclinic.project(H)`, which computes `LEAD_MATRIX @ H`. The frontal axis $\alpha = \operatorname{atan2}(H_y, H_x)$ is `vclinic.frontal_axis(V_I, V_aVF)`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 4 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(4)

In [ ]:
# The lead vectors of the virtual clinic: x to the left, y down, z to the front
lead_table = pd.DataFrame(vclinic.LEAD_MATRIX, index=vclinic.LEADS,
                          columns=["Lx", "Ly", "Lz"]).round(3)
lead_table

In [ ]:
# The worked example of the lecture
H = np.array([0.6, 0.8, -0.2])           # heart vector (mV)
V = vclinic.project(H)
volts = pd.Series(V, index=vclinic.LEADS).round(3)
print(volts.to_string())
print("Einthoven: I + III =", round(volts["I"] + volts["III"], 3), "and II =", volts["II"])
print("Goldberger: aVR + aVL + aVF =", round(volts["aVR"] + volts["aVL"] + volts["aVF"], 3))
print("Frontal axis (degrees):", round(vclinic.frontal_axis(V[0], V[5]), 1))

**Python note.** `@` multiplies a matrix by a vector. Each row of the 12 x 3 lead matrix is multiplied element by element with the three numbers of `H`, and the products are added. A `pd.Series` is a single labelled column, here indexed by the lead names.

In [ ]:
# The 12-lead ECG of the case
fig = teach.plot_12lead(case.time, case.leads, title="VC-0317")
plt.show()
L = case.leads
print("Largest deviation from Einthoven's law (mV):",
      float(np.abs(L["II"] - (L["I"] + L["III"])).max()))

**Explore.** Turn the heart vector in the frontal plane and toward the front or the back of the chest, and watch the twelve projections. A lead shows its largest positive deflection when the vector points along it and a negative one when the vector points away.

In [ ]:
# Explore: one vector, twelve projections
def show_projection(frontal=50, anterior=-15, size=1.3):
    h = size * vclinic.direction(frontal, anterior)
    v = vclinic.project(h)
    colors = ["#9B1B1B" if x >= 0 else "#1F5F8B" for x in v]
    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.bar(vclinic.LEADS, v, color=colors)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_ylim(-2.5, 2.5)
    ax.set_ylabel("Projection (mV)")
    ax.set_title(f"H = ({h[0]:.2f}, {h[1]:.2f}, {h[2]:.2f}) mV, axis "
                 f"{vclinic.frontal_axis(v[0], v[5]):.0f}°")
    plt.show()


try:
    from ipywidgets import interact, IntSlider, FloatSlider
    interact(show_projection,
             frontal=IntSlider(value=50, min=-180, max=180, step=5, description="frontal (°)"),
             anterior=IntSlider(value=-15, min=-90, max=90, step=5, description="front/back (°)"),
             size=FloatSlider(value=1.3, min=0.2, max=2.0, step=0.1, description="size (mV)"))
except ImportError:
    show_projection(50, -15, 1.3)

**Exercise E.** Compute the voltage of lead aVL for $\mathbf{H} = (0.6, 0.8, -0.2)$ mV with the lead vector of aVL from `lead_table`, and round to three decimals.

<details><summary><b>Prompt route</b> for Exercise E: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: `vclinic.LEAD_VECTORS` is a dictionary that maps each lead name, "I", "II", "III", "aVR", "aVL", "aVF" and "V1" to "V6", to a NumPy array of three numbers. The axes point to the left (x), to the feet (y) and to the front (z). The voltage of a lead in millivolts is the dot product of its lead vector with the heart vector H.
Task: Compute the voltage of lead aVL for H = (0.6, 0.8, -0.2) mV.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: Round the voltage to three decimals and convert it to a float. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise E", answer)`. Return only the code in one block, with a short comment above each step.
Test: Lead I has the vector (1, 0, 0), so the same code must give 0.6 mV for lead I.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise E", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
l_avl = vclinic.LEAD_VECTORS["aVL"]
answer = round(float(l_avl @ np.array([0.6, 0.8, -0.2])), 3)
teach.check("Exercise E", answer)

**Observe and reflect.** The same vector gives a positive deflection in lead II and a negative one in lead V1. Explain the difference with the angle between the vector and each lead. Why do the chest leads show larger deflections than the limb leads for vectors of the same size?

**Prompt challenge, Section 4: The frontal axis of the whole clinic.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: The DataFrame `clinic` with the column ecg_axis, the frontal QRS axis in degrees.
Task: Draw a histogram of ecg_axis with vertical lines at -30° and +90°. Print the number and the percentage of patients with left axis deviation (axis below -30°) and with right axis deviation (axis above +90°).
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure and four numbers. Return only the code in one block, with a short comment above each step.
Test: The counts of the three groups, left, normal and right, must add up to 500.
</pre>

</details>

**What to check.** Six patients (1.2 percent) have left axis deviation and 25 patients (5.0 percent) right axis deviation.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 5. Measuring PR, QRS and QT

*Lecture: [Intervals: PR, QRS and QT](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#intervals-pr-qrs-and-qt)*

**From the formulas to the code.** The heart rate $HR = 60 / RR$ is `60 / rr_s`. Bazett's correction $QT_{cB} = QT / \sqrt{RR}$ is `qt / np.sqrt(rr_s)`, and Fridericia's $QT_{cF} = QT / \sqrt[3]{RR}$ is `qt / np.cbrt(rr_s)`; both need the RR interval in seconds. The end of the T wave by the tangent method, $t_{end} = t_s + (b - x(t_s)) / x'(t_s)$, is the line `t_s + (base - x[t_s]) / slope * fs` in the function `t_end_tangent`.

The section builds the measurement in five steps and then joins them in one function, `measure_ecg`. The same steps are kept in `vclinic.measure` for the later weeks.

**Started here?** Run the catch-up cell below once. It runs the code that Section 5 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(5)

In [ ]:
# Step 1. R peaks: band-pass, derivative, squaring and a 150 ms moving window
from scipy.signal import butter, filtfilt, find_peaks


def bandpass(x, fs, low, high):
    b, a = butter(2, [low / (fs / 2), high / (fs / 2)], btype="band")
    return filtfilt(b, a, x)


def lowpass(x, fs, cutoff=20.0):
    b, a = butter(2, cutoff / (fs / 2), btype="low")
    return filtfilt(b, a, x)


def detect_r_peaks(x, fs):
    y = bandpass(x, fs, 5, 15)                 # keeps the QRS frequencies
    d = np.gradient(y) * fs                    # slope (mV/s)
    z = d ** 2                                 # only large slopes stand out
    w = int(round(0.150 * fs))                 # 150 ms window
    m = np.convolve(z, np.ones(w) / w, mode="same")
    peaks, _ = find_peaks(m, height=0.3 * np.percentile(m, 99),
                          distance=int(0.25 * fs))
    yy = bandpass(x, fs, 1, 40)
    half = int(0.075 * fs)
    r = []
    for p in peaks:                            # largest deflection within 75 ms
        lo, hi = max(0, p - half), min(len(x), p + half + 1)
        r.append(lo + int(np.argmax(np.abs(yy[lo:hi]))))
    return np.unique(r)


r_idx = detect_r_peaks(case.lead("II"), case.fs)
print("R peaks found:", len(r_idx))
print("times (s):", np.round(r_idx / case.fs, 3))
print("true R times (s):", np.round(case.truth["all_beats_s"], 3))

In [ ]:
# Step 2. QRS onset and offset from the slope of all 12 leads
def global_slope(leads, fs):
    arr = np.column_stack([bandpass(leads[:, j], fs, 1, 40)
                           for j in range(leads.shape[1])])
    g = np.abs(np.gradient(arr, axis=0)).sum(axis=1) * fs
    w = max(1, int(round(0.010 * fs)))
    return np.convolve(g, np.ones(w) / w, mode="same")


def qrs_bounds(leads, fs, r_idx, frac=0.08):
    g = global_slope(leads, fs)
    onsets, offsets = [], []
    for r in r_idx:
        lo, hi = max(0, r - int(0.12 * fs)), min(len(g) - 1, r + int(0.14 * fs))
        level = frac * g[lo:hi + 1].max()      # 8 percent of the largest slope
        on = r
        while on > lo and g[on] > level:
            on -= 1
        off = r
        while off < hi and g[off] > level:
            off += 1
        onsets.append(on)
        offsets.append(off)
    return np.array(onsets), np.array(offsets)


on_idx, off_idx = qrs_bounds(case.leads.to_numpy(), case.fs, r_idx)
qrs_ms = (off_idx - on_idx) / case.fs * 1000
print("QRS durations (ms):", np.round(qrs_ms, 0))

**Python note.** `while` repeats its indented lines as long as its condition holds. Here it walks backward from the R peak, one sample at a time, until the slope of the 12 leads falls below the level. `np.column_stack` places arrays side by side as the columns of a matrix.

In [ ]:
# Step 3. End of the T wave by the tangent method (lead II, low-passed at 20 Hz)
def t_end_tangent(x, fs, qrs_off, next_r, base):
    lo = qrs_off + int(0.08 * fs)
    hi = qrs_off + int(0.7 * (next_r - qrs_off))
    seg = x[lo:hi] - base
    k = int(np.argmax(np.abs(seg)))
    peak = lo + k                              # T peak
    sign = 1.0 if seg[k] >= 0 else -1.0
    d = np.gradient(x) * fs
    stop = min(len(x) - 1, peak + int(0.20 * fs))
    t_s = peak + int(np.argmin(sign * d[peak:stop]))   # steepest point
    slope = d[t_s]
    if slope == 0:
        return np.nan
    return t_s + (base - x[t_s]) / slope * fs  # where the tangent meets the baseline


x20 = lowpass(case.lead("II"), case.fs, 20)
fs = case.fs
i = 2                                          # third beat
base = np.median(x20[on_idx[i] - int(0.03 * fs):on_idx[i]])
t_end = t_end_tangent(x20, fs, off_idx[i], r_idx[i + 1], base)
print(f"QT of beat: {(t_end - on_idx[i]) / fs * 1000:.1f} ms, true: {case.truth['qt_ms']:.0f} ms")

In [ ]:
# The tangent construction on the third beat
d20 = np.gradient(x20) * fs
seg = x20[off_idx[i] + int(0.08 * fs):off_idx[i] + int(0.7 * (r_idx[i + 1] - off_idx[i]))]
peak = off_idx[i] + int(0.08 * fs) + int(np.argmax(np.abs(seg - base)))
t_s = peak + int(np.argmin(d20[peak:peak + int(0.2 * fs)]))
tt = np.linspace(t_s / fs - 0.05, t_end / fs + 0.02, 50)
fig, ax = plt.subplots(figsize=(9, 4))
w = slice(on_idx[i] - int(0.1 * fs), int(t_end) + int(0.12 * fs))
teach.plot_strip(ax, case.time[w], x20[w], label="II", v_range=(-0.5, 1.5))
ax.plot(tt, x20[t_s] + d20[t_s] * (tt - t_s / fs), color="#9B1B1B", lw=1.5,
        label="tangent at the steepest point")
ax.axhline(base, color="#1F5F8B", lw=1, ls="--", label="baseline")
ax.axvline(on_idx[i] / fs, color="k", lw=1, ls=":")
ax.axvline(t_end / fs, color="#9B1B1B", lw=1, ls=":")
ax.legend(loc="upper right", fontsize=8)
plt.show()

In [ ]:
# Step 4. Onset of the P wave
def p_onset(x, fs, qrs_on, earliest=0, frac=0.10):
    lo = max(0, int(earliest), qrs_on - int(0.30 * fs))
    hi = qrs_on - int(0.04 * fs)
    if hi - lo < int(0.08 * fs):               # too short to hold a P wave
        return np.nan
    seg = x[lo:hi]
    base = np.median(seg[:int(0.04 * fs)])     # flat segment before the P wave
    seg = seg - base
    k = int(np.argmax(seg))
    j = k
    while j > 0 and seg[j] > frac * seg[k]:    # back to 10 percent of the P height
        j -= 1
    return lo + j


po = p_onset(x20, fs, on_idx[i], earliest=t_end_tangent(
    x20, fs, off_idx[i - 1], r_idx[i], base) + 0.02 * fs)
print(f"PR of beat: {(on_idx[i] - po) / fs * 1000:.0f} ms, true: {case.truth['pr_ms']:.0f} ms")

In [ ]:
# Step 5. All steps in one function: the measurement of the ECG laboratory
def measure_ecg(leads, fs, lead="II", sinus=True):
    """Median heart rate, PR, QRS, QT and QTc of a 12-lead recording."""
    leads = np.asarray(leads, dtype=float)
    x = leads[:, ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3",
                  "V4", "V5", "V6"].index(lead)]
    r = detect_r_peaks(leads[:, 1], fs)
    on, off = qrs_bounds(leads, fs, r)
    x20 = lowpass(x, fs, 20)
    pr, qrs, qt = [], [], []
    t_prev = None
    for k in range(len(r) - 1):
        base = np.median(x20[max(0, on[k] - int(0.03 * fs)):on[k]])
        te = t_end_tangent(x20, fs, off[k], r[k + 1], base)
        qrs.append((off[k] - on[k]) / fs * 1000)
        qt.append((te - on[k]) / fs * 1000)
        if sinus and t_prev is not None and np.isfinite(t_prev):
            pr.append((on[k] - p_onset(x20, fs, on[k], t_prev + 0.02 * fs)) / fs * 1000)
        t_prev = te
    rr_s = np.mean(np.diff(r)) / fs
    out = {"hr_bpm": 60 / rr_s, "rr_s": rr_s,
           "pr_ms": np.nanmedian(pr) if np.isfinite(pr).any() else np.nan,
           "qrs_ms": np.median(qrs), "qt_ms": np.nanmedian(qt)}
    out["qtc_bazett_ms"] = out["qt_ms"] / np.sqrt(rr_s)
    out["qtc_fridericia_ms"] = out["qt_ms"] / np.cbrt(rr_s)
    return out


def compare(ecg):
    m = measure_ecg(ecg.leads.to_numpy(), ecg.fs, sinus=ecg.truth["rhythm"] == "sinus")
    keys = ["hr_bpm", "pr_ms", "qrs_ms", "qt_ms", "qtc_bazett_ms", "qtc_fridericia_ms"]
    return pd.DataFrame({"measured": [m[k] for k in keys],
                         "true": [ecg.truth[k] for k in keys]},
                        index=keys).round(1)


compare(case)

**Observe and reflect.** For the case of the week, Bazett's formula gives a QTc above the threshold of 460 ms for women. Fridericia's formula gives a value well below it. At 85 bpm, which correction would you report, and why? The lecture section on QT correction gives the recommendation of the 2009 statement.

In [ ]:
# The five patients of the interactive lab, Part C
LAB_PATIENTS = ["VC-0317", "VC-0079", "VC-0254", "VC-0188", "VC-0018"]
rows = []
for pid in LAB_PATIENTS:
    e = vclinic.patient_ecg(clinic, pid)
    m = measure_ecg(e.leads.to_numpy(), e.fs, sinus=e.truth["rhythm"] == "sinus")
    rows.append({"patient": pid, "conduction": e.truth["conduction"],
                 "HR": e.truth["hr_bpm"], "PR_true": e.truth["pr_ms"], "PR_alg": m["pr_ms"],
                 "QRS_true": e.truth["qrs_ms"], "QRS_alg": m["qrs_ms"],
                 "QT_true": e.truth["qt_ms"], "QT_alg": m["qt_ms"]})
lab_table = pd.DataFrame(rows).set_index("patient").round(1)
lab_table

The last patient, VC-0018, has a left bundle branch block (LBBB). Its QRS is wide, and its T wave in lead II is small and opposite to the main QRS direction. The tangent method in lead II is therefore unreliable for this patient, and the advanced path below shows why.

In [ ]:
# Validation on 100 patients of the clinic (sinus rhythm, normal conduction)
sample = clinic[(clinic["ecg_rhythm"] == "sinus")
                & (clinic["ecg_conduction"] == "normal")].head(100)
errors, qt_pairs = [], []
for pid in sample["patient_id"]:
    e = vclinic.patient_ecg(clinic, pid)
    m = measure_ecg(e.leads.to_numpy(), e.fs)
    errors.append({k: m[k] - e.truth[k] for k in ["pr_ms", "qrs_ms", "qt_ms"]})
    qt_pairs.append((m["qt_ms"], e.truth["qt_ms"]))
errors = pd.DataFrame(errors)
qt_pairs = np.array(qt_pairs)
summary = pd.DataFrame({"bias (ms)": errors.mean(),
                        "SD (ms)": errors.std(),
                        "largest ": errors.abs().max()}).round(1)
summary

**Your calipers against the algorithm and the truth.** The tab *Reflect and export* of the interactive lab exports your caliper measurements of Part C as `my_caliper_measurements.csv`. Save the file in the folder of this notebook and run the next cell. Where a download is not possible, the button *Copy caliper measurements* copies the same text: Paste it between the triple quotes of `PASTED_CALIPERS` in the next cell. Without your own measurements, the cell uses an example file of simulated manual measurements.

In [ ]:
# Bland-Altman analysis of the QT interval
import io
import os
from vclinic.datasets import data_path

# paste the copied caliper measurements between the triple quotes
PASTED_CALIPERS = """
"""

own = "my_caliper_measurements.csv"
if PASTED_CALIPERS.strip():
    source = "PASTED_CALIPERS"
    calipers = pd.read_csv(io.StringIO(PASTED_CALIPERS.strip()))
elif os.path.exists(own):
    source = own
    calipers = pd.read_csv(own)
else:
    source = data_path("examples", "example_caliper_measurements.csv")
    calipers = pd.read_csv(source)
print("Measurements from:", source)
qt_manual = calipers[calipers["measure"] == "QT"].set_index("patient_id")["value_ms"]
pts = [p for p in qt_manual.index if p != "VC-0018"]
truth_qt = [vclinic.patient_ecg(clinic, p).truth["qt_ms"] for p in pts]
alg_qt = [lab_table.loc[p, "QT_alg"] for p in pts]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
teach.plot_bland_altman(axes[0], qt_manual[pts].to_numpy(), truth_qt)
axes[0].set_title("Calipers minus truth")
teach.plot_bland_altman(axes[1], qt_pairs[:, 0], qt_pairs[:, 1])
axes[1].set_title("Algorithm minus truth, 100 patients")
plt.tight_layout()
plt.show()

**Python note.** `elif` means *else if*: Python tests the conditions from top to bottom and runs only the first block whose condition is true. `io.StringIO` makes a piece of text behave like a file, so `pd.read_csv` can read pasted text in the same way as a file on disk.

**Exercise F.** Patient VC-0254 has a QT of 343 ms at a heart rate of 98 bpm in the registry. Compute its QTc with Fridericia's formula from these two numbers and round to one decimal.

<details><summary><b>Prompt route</b> for Exercise F: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: A QT interval of 343 ms at a heart rate of 98 bpm.
Task: Compute the QTc with Fridericia's formula, QTc = QT / RR^(1/3), with the RR interval in seconds, RR = 60 / HR.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: Round the QTc to one decimal. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise F", answer)`. Return only the code in one block, with a short comment above each step.
Test: At 60 bpm the RR interval is 1 s, so the QTc must equal the QT.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise F", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(343 / np.cbrt(60 / 98), 1)
teach.check("Exercise F", answer)

**Exercise G.** Measure the PR interval of patient VC-0188 with `measure_ecg` and round it to the nearest millisecond. Does the result meet the definition of a first-degree atrioventricular block?

<details><summary><b>Prompt route</b> for Exercise G: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: The notebook defines `measure_ecg(leads, fs)`. It takes a NumPy array of shape (samples, 12) with the twelve leads in millivolts, in the order I, II, III, aVR, aVL, aVF, V1 to V6, and the sampling frequency in Hz. It returns a dictionary with the keys "hr_bpm", "pr_ms", "qrs_ms", "qt_ms", "qtc_bazett_ms" and "qtc_fridericia_ms". `vclinic.patient_ecg(clinic, patient_id)` returns an ECG object, called `e` here. `e.leads` is a pandas DataFrame of the twelve leads, and `e.fs` is the sampling frequency in Hz.
Task: Measure the PR interval of patient "VC-0188" and decide whether it meets the definition of a first-degree atrioventricular block, a PR interval above 200 ms.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: Round the PR interval to the nearest millisecond and print the decision. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise G", answer)`. Return only the code in one block, with a short comment above each step.
Test: Run the same code for patient "VC-0317", whose PR interval in the registry is 190 ms; the measurement should lie within 10 ms of it.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise G", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
e = vclinic.patient_ecg(clinic, "VC-0188")
answer = round(measure_ecg(e.leads.to_numpy(), e.fs)["pr_ms"])
teach.check("Exercise G", answer)
print("First-degree block (PR > 200 ms):", answer > 200)

**Advanced path.** The 2009 recommendations advise measuring the QT in the lead with the longest QT, which is often V2 or V3. Extend `measure_ecg` so that it measures the QT in leads II, V2, V3 and V5 and reports the longest. Try it on VC-0018 and look at the T waves of each lead: Why does the tangent method struggle when the T wave is opposite to the QRS?

In [ ]:
# Advanced path: the QT in four leads
e = vclinic.patient_ecg(clinic, "VC-0018")
for lead in ["II", "V2", "V3", "V5"]:
    print(lead, round(measure_ecg(e.leads.to_numpy(), e.fs, lead=lead)["qt_ms"], 1))
print("true QT:", e.truth["qt_ms"])

**Prompt challenge, Section 5: Bazett against Fridericia across heart rates.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: The DataFrame `clinic` with the columns ecg_hr, the heart rate in bpm, and ecg_qt, the QT interval in ms.
Task: For the first 50 patients, compute the QTc by Bazett, QT / √RR, and by Fridericia, QT / RR^(1/3), with RR = 60 / ecg_hr in seconds. Plot the difference Bazett minus Fridericia against the heart rate, with a horizontal line at zero, and print the largest difference.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure and one number. Return only the code in one block, with a short comment above each step.
Test: At exactly 60 bpm both formulas give the same QTc, so the difference is zero.
</pre>

</details>

**What to check.** The difference is positive in every patient above 60 bpm and negative in the three patients below 60 bpm. The largest difference is 48.6 ms.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 6. Finding every beat in a real recording: MIT-BIH record 100

*Lecture: [Finding every beat: R-peak detection](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#finding-every-beat-r-peak-detection)*

**From the formulas to the code.** The width of the integration window, $N = 0.150 \cdot f_s$, is `int(round(0.150 * fs))`, which is 54 samples at 360 Hz. Sensitivity $Se = TP / (TP + FN)$ and positive predictive value $PPV = TP / (TP + FP)$ are computed from the counts that `vclinic.measure.match_beats` returns. A detected beat counts as true when it lies within 150 ms of a reference beat.

Record 100 of the Massachusetts Institute of Technology and Beth Israel Hospital (MIT-BIH) Arrhythmia Database lasts 30 minutes and holds two channels at 360 Hz. Cardiologists annotated every beat. The record is bundled with the course, so the section runs without an internet connection.

**Started here?** Run the catch-up cell below once. It runs the code that Section 6 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(6)

In [ ]:
from vclinic.datasets import read_physionet, read_physionet_annotation
from vclinic.measure import match_beats

mit, fs_mit, notes = read_physionet("100", "mitdb")
ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb")
print(mit.shape, fs_mit, notes)
print(pd.Series(ann_symbols).value_counts())
mit.head()

The annotation symbols follow the convention of PhysioNet: `N` is a normal beat, `A` an atrial premature beat and `V` a premature ventricular contraction. `+` is not a beat; it marks a change of rhythm. The `wfdb` package reads the record when it is installed; otherwise the small reader of `vclinic.physionet` does the same work.

In [ ]:
# The four stages of the detector on the first 3 seconds
x = mit["MLII"].to_numpy()
seg = slice(0, int(3 * fs_mit))
y = bandpass(x, fs_mit, 5, 15)
z = (np.gradient(y) * fs_mit) ** 2
w = int(round(0.150 * fs_mit))
m = np.convolve(z, np.ones(w) / w, mode="same")
t = np.arange(len(x)) / fs_mit
fig, axes = plt.subplots(4, 1, figsize=(11, 7), sharex=True)
names = ["raw lead MLII (mV)", "band-pass 5-15 Hz",
         "squared slope", f"moving window ({w} samples)"]
for ax, sig, name in zip(axes, [x, y, z, m], names):
    ax.plot(t[seg], sig[seg], color="#9B1B1B" if ax is axes[3] else "k", lw=0.9)
    ax.set_ylabel(name, fontsize=8)
axes[3].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

In [ ]:
# Detection on the whole recording and comparison with the annotations
beat_symbols = set("NLRBAaJSVrFejnE/fQ?")
ref = np.array([s for s, sym in zip(ann_samples, ann_symbols) if sym in beat_symbols])
found = detect_r_peaks(x, fs_mit)
tp, fp, fn = match_beats(found / fs_mit, ref / fs_mit, window_s=0.150)
print(f"reference beats: {len(ref)}, detected: {len(found)}")
print(f"TP = {tp}, FP = {fp}, FN = {fn}")
print(f"Se = {100 * tp / (tp + fn):.2f} %, PPV = {100 * tp / (tp + fp):.2f} %")

**Exercise H.** How many atrial premature beats (symbol `A`) does record 100 hold?

<details><summary><b>Prompt route</b> for Exercise H: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: A Python list named `ann_symbols` holds the annotation symbols of MIT-BIH record 100, one per annotation: "N" for a normal beat, "A" for an atrial premature beat, "V" for a premature ventricular contraction and "+" for a rhythm change.
Task: Count the atrial premature beats.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The number of beats as an integer. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise H", answer)`. Return only the code in one block, with a short comment above each step.
Test: Count every symbol with collections.Counter and print the counts; their sum must equal len(ann_symbols).
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise H", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = ann_symbols.count("A")
teach.check("Exercise H", answer)

**Observe and reflect.** The detector finds every beat of record 100. Why is this result not enough to trust it on every recording? Think of noise, of wide ventricular beats and of tall T waves. Which records of a test set would you choose to challenge it?

**Advanced path.** With an internet connection, `wfdb.rdrecord("105", pn_dir="mitdb")` and `wfdb.rdann("105", "atr", pn_dir="mitdb")` read record 105, which is known for its noise, straight from PhysioNet. Run the detector on it and compare the result with `neurokit2.ecg_peaks` if neurokit2 is installed.

In [ ]:
# Advanced path: record 105 from PhysioNet (needs internet and wfdb)
try:
    import wfdb
    rec = wfdb.rdrecord("105", pn_dir="mitdb", sampto=360 * 300)
    ann = wfdb.rdann("105", "atr", pn_dir="mitdb", sampto=360 * 300)
    ref105 = np.array([s for s, sym in zip(ann.sample, ann.symbol) if sym in beat_symbols])
    found105 = detect_r_peaks(rec.p_signal[:, 0], rec.fs)
    print(match_beats(found105 / rec.fs, ref105 / rec.fs))
except Exception as err:
    print("Not available here:", type(err).__name__)

**Prompt challenge, Section 6: Heart rate over thirty minutes.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: `found` holds the detected R peaks of MIT-BIH record 100 as sample positions, and `fs_mit` its sampling frequency, 360 Hz.
Task: Compute the heart rate of every beat, 60 / RR with the RR interval in seconds, and plot it against time in minutes. Print the mean heart rate.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure and one number. Return only the code in one block, with a short comment above each step.
Test: An RR interval of 360 samples at 360 Hz lasts 1 s, that is, 60 bpm.
</pre>

</details>

**What to check.** The mean heart rate is about 75.8 bpm, and the beat-to-beat rate ranges from about 53 to 115 bpm.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 7. Heart rate variability

*Lecture: [Heart rate variability](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#heart-rate-variability)*

**From the formulas to the code.** With the RR intervals in milliseconds in an array `rr_ms`, the standard deviation of the normal-to-normal intervals, $SDNN = \sqrt{\frac{1}{N-1}\sum (RR_i - \overline{RR})^2}$, is `np.std(rr_ms, ddof=1)`. The root mean square of successive differences, $RMSSD = \sqrt{\frac{1}{N-1}\sum (RR_{i+1} - RR_i)^2}$, is `np.sqrt(np.mean(np.diff(rr_ms) ** 2))`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 7 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(7)

In [ ]:
# The worked example of the lecture: six RR intervals (ms)
rr_ms = np.array([812, 845, 830, 798, 860, 841])
print("mean RR:", rr_ms.mean())
print("SDNN:", round(np.std(rr_ms, ddof=1), 2), "ms")
print("successive differences:", np.diff(rr_ms))
print("RMSSD:", round(np.sqrt(np.mean(np.diff(rr_ms) ** 2)), 2), "ms")

In [ ]:
# Five minutes of lead II for the case, as the 1996 Task Force recommends for short-term HRV
from vclinic.ecg import lead_ii_long

t5, x5, truth5 = lead_ii_long(case.params, duration_s=300, fs=250, seed=17)
r5 = detect_r_peaks(x5, 250)
rr5 = np.diff(r5) / 250 * 1000                 # RR intervals in ms
sdnn = np.std(rr5, ddof=1)
rmssd = np.sqrt(np.mean(np.diff(rr5) ** 2))
print(f"beats: {len(r5)}, mean HR: {60000 / rr5.mean():.1f} bpm, SDNN: {sdnn:.1f} ms, RMSSD: {rmssd:.1f} ms")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(r5[1:] / 250, rr5, color="#9B1B1B", lw=0.8)
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("RR (ms)")
axes[0].set_title("Tachogram")
axes[1].scatter(rr5[:-1], rr5[1:], s=6, color="#1F5F8B")
axes[1].set_xlabel("RR$_i$ (ms)")
axes[1].set_ylabel("RR$_{i+1}$ (ms)")
axes[1].set_title("Poincaré plot")
plt.tight_layout()
plt.show()

In [ ]:
# Record 100: normal-to-normal intervals of the first 5 minutes
pairs = [(s1, s2) for (s1, a), (s2, b) in zip(zip(ann_samples, ann_symbols),
                                               zip(ann_samples[1:], ann_symbols[1:]))
         if a == "N" and b == "N" and s2 <= 300 * fs_mit]
nn_ms = np.array([(s2 - s1) / fs_mit * 1000 for s1, s2 in pairs])
print(f"NN intervals: {len(nn_ms)}, mean HR: {60000 / nn_ms.mean():.1f} bpm")
print(f"SDNN: {np.std(nn_ms, ddof=1):.1f} ms, RMSSD: {np.sqrt(np.mean(np.diff(nn_ms) ** 2)):.1f} ms")

**Explore.** The generator of the virtual clinic has a variability parameter `hrv`: 1.0 is a typical adult at rest, and smaller values mimic older age or heart failure. Move it and watch SDNN and RMSSD of a 5-minute recording.

In [ ]:
# Explore: variability and the two measures
def show_hrv(hrv=1.0, heart_rate=70):
    params = dict(case.params, hrv=hrv, hr_bpm=heart_rate)
    _, xs, _ = lead_ii_long(params, duration_s=300, fs=250, seed=5)
    rr = np.diff(detect_r_peaks(xs, 250)) / 250 * 1000
    print(f"hrv = {hrv:.2f}, HR = {heart_rate} bpm -> SDNN = {np.std(rr, ddof=1):.1f} ms, "
          f"RMSSD = {np.sqrt(np.mean(np.diff(rr) ** 2)):.1f} ms")


try:
    from ipywidgets import interact, FloatSlider, IntSlider
    interact(show_hrv, hrv=FloatSlider(value=1.0, min=0.2, max=1.6, step=0.1),
             heart_rate=IntSlider(value=70, min=50, max=110, step=5, description="HR"))
except ImportError:
    show_hrv(1.0, 70)
    show_hrv(0.4, 70)

**Exercise I.** Compute the RMSSD of the RR intervals 1000, 960, 1010, 990 and 1030 ms, and round it to two decimals.

<details><summary><b>Prompt route</b> for Exercise I: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: The RR intervals 1000, 960, 1010, 990 and 1030 ms.
Task: Compute the root mean square of successive differences (RMSSD): Take the differences between neighbouring intervals, square them, average the squares and take the square root.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: Round the RMSSD to two decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise I", answer)`. Return only the code in one block, with a short comment above each step.
Test: The intervals 800, 800 and 800 ms must give 0, and the intervals 800, 820 and 800 ms must give 20.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise I", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
rr_ex = np.array([1000, 960, 1010, 990, 1030])
answer = round(float(np.sqrt(np.mean(np.diff(rr_ex) ** 2))), 2)
teach.check("Exercise I", answer)

**Observe and reflect.** RMSSD reacts mainly to fast, beat-to-beat changes, and SDNN also to slow ones. Why does the 1996 Task Force recommend recordings of 5 minutes for short-term measures, and why should values from recordings of different lengths not be compared?

**Prompt challenge, Section 7: A Poincaré plot.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: `rr5` holds the RR intervals in milliseconds of five minutes of lead II of the case of the week.
Task: Draw a Poincaré plot, each RR interval against the next one, with equal axes and the identity line. Print SDNN, the standard deviation with ddof=1, and RMSSD.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure and two numbers. Return only the code in one block, with a short comment above each step.
Test: For the intervals 800, 820 and 800 ms, the RMSSD is 20 ms.
</pre>

</details>

**What to check.** SDNN and RMSSD equal the values printed earlier in Section 7, 17.7 ms and 17.4 ms.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 8. A real 12-lead ECG from the PTB Diagnostic database

*Lecture: [Real data: PhysioNet records](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#real-data-physionet-records)*

Record `s0010_re` of patient 1 of the PTB Diagnostic ECG Database, from the Physikalisch-Technische Bundesanstalt (PTB) in Germany, holds the 12 standard leads and three Frank leads at 1000 Hz for 38.4 seconds. Its header describes the patient. The record shows how the steps of Section 5 behave on a real recording.

**Started here?** Run the catch-up cell below once. It runs the code that Section 8 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(8)

In [ ]:
ptb, fs_ptb, header = read_physionet("s0010_re", "ptbdb/patient001")
print(ptb.shape, fs_ptb)
for line in header[:9]:
    print(line)

In [ ]:
# The twelve leads of the real record (column names are lower case)
names = {"I": "i", "II": "ii", "III": "iii", "aVR": "avr", "aVL": "avl", "aVF": "avf",
         "V1": "v1", "V2": "v2", "V3": "v3", "V4": "v4", "V5": "v5", "V6": "v6"}
fig = teach.plot_12lead(ptb.index.to_numpy(), ptb, title="PTB patient001, s0010_re",
                        v_range=(-1.5, 1.5), names=names)
plt.show()

In [ ]:
# Einthoven's and Goldberger's relations on real data
print("max |II - (I + III)| =", round(float(np.abs(ptb["ii"] - (ptb["i"] + ptb["iii"])).max()), 4), "mV")
print("max |aVR + aVL + aVF| =", round(float(np.abs(ptb["avr"] + ptb["avl"] + ptb["avf"]).max()), 4), "mV")

In [ ]:
# The measurement of Section 5 on the real record
leads_ptb = ptb[[names[k] for k in vclinic.LEADS]].to_numpy()
m_ptb = measure_ecg(leads_ptb, fs_ptb)
pd.Series(m_ptb).round(1)

**Observe and reflect.** The algorithm measures a QRS duration of about 137 ms in this record. Look at lead V1 and at the inferior leads in the figure. Which pattern could explain a QRS of this width, and what do the inferior leads show? The 2009 recommendations ask that computer measurements be checked by eye: Measure one beat yourself on the figure and compare.

**Advanced path.** The three Frank leads `vx`, `vy` and `vz` record the heart vector almost directly. Plot the QRS loop of one beat in three dimensions. Textbooks use different sign conventions for the Frank leads, so the loop is drawn here as recorded.

In [ ]:
# Advanced path: the QRS loop of one beat from the Frank leads
r_ptb = detect_r_peaks(ptb["ii"].to_numpy(), fs_ptb)
k = r_ptb[3]
loop = ptb[["vx", "vy", "vz"]].to_numpy()[k - int(0.06 * fs_ptb):k + int(0.08 * fs_ptb)]
fig = plt.figure(figsize=(5.5, 5))
ax = fig.add_subplot(projection="3d")
ax.plot(loop[:, 0], loop[:, 1], loop[:, 2], color="#9B1B1B")
ax.set_xlabel("vx (mV)")
ax.set_ylabel("vy (mV)")
ax.set_zlabel("vz (mV)")
plt.show()

**Prompt challenge, Section 8: The chest leads of a real infarction.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: `ptb` is a pandas DataFrame with the PTB record s0010_re: The columns v1 to v6 hold the chest leads in millivolts, sampled at `fs_ptb`, 1000 Hz.
Task: Plot the first 3 seconds of V1 to V6 one under the other, with a shared time axis in seconds and the voltage in millivolts.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure with six rows. Return only the code in one block, with a short comment above each step.
Test: Three seconds at 1000 Hz hold 3000 samples per lead.
</pre>

</details>

**What to check.** The figure has six rows of 3 s each. The header of the record states an acute inferolateral infarction: Which chest leads face the lateral wall?

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 9. AI pair programming: The code safety checklist

*Lecture: [AI pair programming, part 1: The code safety checklist](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#ai-pair-programming-part-1-the-code-safety-checklist)*

An AI assistant can write most of the code of this course, and it can also write code that runs without error and returns a wrong number. The course therefore treats AI-written code like a new member of the team: It is welcome, and it is checked. The prompt card below gives the structure of a good request.

**Prompt card.** *Role:* You are helping a cardiologist who is learning Python. *Context:* A Jupyter notebook with NumPy imported as np. *Data:* A 12-lead ECG of 10 seconds, sampled at 500 Hz, stored in a pandas DataFrame `leads` with one column per lead, in millivolts. The QT and RR intervals are given in samples. *Task:* Write a function that returns the QTc with Bazett's formula from the QT and RR intervals in samples, and apply it to QT = 196 samples and RR = 353 samples. *Constraints:* Use only NumPy; take the sampling frequency as an argument without a default value. *Output:* The QTc in milliseconds, rounded to one decimal. *Test:* QT = 200 samples and RR = 500 samples at 500 Hz must give 400.0 ms, because the QT is then 400 ms and the RR interval 1 s.

**Check before using.** The lecture page gives the code safety checklist of the course: *sign in* before running (units, sampling frequency, lead, no patient identifier), *time out* while running (errors, physiological values) and *sign out* before using (comparison with a known truth, every line explained, prompt and changes logged).

An AI assistant produced the function below from a shorter prompt that gave no sampling frequency.

**Started here?** Run the catch-up cell below once. It runs the code that Section 9 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(9)

In [ ]:
# Code written by an AI assistant from a short prompt
def qtc_from_samples(qt_samples, rr_samples, fs=360):
    qt_ms = qt_samples / fs * 1000
    rr_ms = rr_samples / fs * 1000
    return qt_ms / np.sqrt(rr_ms)


print(qtc_from_samples(196, 353))

**Observe and reflect.** The function runs without an error. Go through the checklist: Which pause catches the result, and what are the two errors? One of them comes from a default value that the prompt never asked for.

**Exercise J.** Correct the function: Take `fs` without a default value and use the RR interval in seconds. Compute the QTc for 196 and 353 samples at 500 Hz and round to one decimal.

<details><summary><b>Prompt route</b> for Exercise J: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: An ECG sampled at fs Hz; the QT interval and the RR interval are given as numbers of samples.
Task: Write a function `qtc_from_samples_fixed(qt_samples, rr_samples, fs)` without a default value for fs. It converts the QT interval to milliseconds, qt_samples / fs × 1000, and the RR interval to seconds, rr_samples / fs, and returns Bazett's QTc = QT / √RR in milliseconds. Use it for 196 and 353 samples at 500 Hz.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: Round the QTc to one decimal. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise J", answer)`. Return only the code in one block, with a short comment above each step.
Test: For 200 and 500 samples at 500 Hz the QT is 400 ms and the RR interval 1 s, so the function must return 400.0.
</pre>

</details>

In [ ]:
def qtc_from_samples_fixed(qt_samples, rr_samples, fs):
    return None   # your corrected formula


answer = qtc_from_samples_fixed(196, 353, 500)
answer = round(answer, 1) if answer is not None else None
teach.check("Exercise J", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def qtc_from_samples_fixed(qt_samples, rr_samples, fs):
    """Bazett QTc (ms) from QT and RR given in samples at fs (Hz)."""
    qt_ms = qt_samples / fs * 1000
    rr_s = rr_samples / fs
    return qt_ms / np.sqrt(rr_s)


answer = round(qtc_from_samples_fixed(196, 353, 500), 1)
teach.check("Exercise J", answer)

**AI usage log.** Each weekly project adds a short log with one row per use of an AI assistant:

| Tool | Prompt (short) | What was checked | What was changed |
|---|---|---|---|
| (name of the assistant) | Bazett QTc from samples | Result 466.5 ms against the true values of VC-0317 | Removed the default fs; RR in seconds |

**Prompt challenge, Section 9: A second opinion on AI-written code.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd and matplotlib.pyplot as plt, and the course package vclinic with its module teach is available.
Data: The function `qtc_from_samples` of this section, which an AI assistant wrote from a short prompt: Paste its code below the prompt.
Task: Review the function with three questions. Are the units right? Is the sampling frequency handled safely? Is the result physiological for a QT of 196 samples and an RR interval of 353 samples at 500 Hz? Then return a corrected function.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: A review in three short points and the corrected function. Return only the code in one block, with a short comment above each step.
Test: The corrected function must return 400.0 ms for 200 and 500 samples at 500 Hz.
</pre>

</details>

**What to check.** The review names two problems: the RR interval in milliseconds instead of seconds, and the default sampling frequency of 360 Hz.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 10. Application key: Another patient, another lead

Change `PATIENT` and `LEAD` and run the cell again. Patients worth a look: VC-0183 (a long QT interval), VC-0018 (left bundle branch block), VC-0207 (a PR interval of 218 ms in the registry) and any patient whose `ecg_rhythm` is `AF`. `clinic[clinic["stemi_territory"] != "none"]` lists the patients with an acute ST-elevation myocardial infarction (STEMI), the topic of Week 5.

**Started here?** Run the catch-up cell below once. It runs the code that Section 10 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(10)

In [ ]:
PATIENT = "VC-0183"
LEAD = "II"

e = vclinic.patient_ecg(clinic, PATIENT)
row = clinic[clinic["patient_id"] == PATIENT].iloc[0]
print(PATIENT, row["age"], row["sex"], row["main_diagnosis"], row["ecg_rhythm"], row["ecg_conduction"])
m = measure_ecg(e.leads.to_numpy(), e.fs, lead=LEAD, sinus=e.truth["rhythm"] == "sinus")
keys = ["hr_bpm", "pr_ms", "qrs_ms", "qt_ms", "qtc_bazett_ms", "qtc_fridericia_ms"]
display(pd.DataFrame({"measured": [m[k] for k in keys],
                      "true": [e.truth[k] for k in keys]}, index=keys).round(1))
fig = teach.plot_12lead(e.time, e.leads, title=PATIENT)
plt.show()

## Application challenges by field

| Field | Challenge |
|---|---|
| Electrophysiology | Generate the ECG of VC-0183 at heart rates from 50 to 110 bpm (change `hr_bpm` in `e.params` and keep `qt_ms` scaled by $RR^{1/3}$). Plot QTc by Bazett and by Fridericia against heart rate and decide which one to report in a QT monitoring protocol. |
| Interventional cardiology | List the patients with a STEMI territory, draw their 12-lead ECGs and name the leads with ST elevation for each territory. Compare with the culprit artery that the territory suggests. |
| Heart failure | Compare SDNN and RMSSD of 5-minute recordings between the HFrEF patients and the patients without heart disease of the clinic. |
| Cardio-oncology | Build a QTc follow-up table for one patient on a QT-prolonging treatment, with serial ECGs at different heart rates, and flag the visits at which the QTc crosses the threshold of your local protocol. |
| Pediatric cardiology | The 2009 recommendations use lower QRS thresholds in children. Change the threshold of a wide QRS in a function of your own and discuss what else would change in a child's ECG. |
| Sports cardiology | Generate a resting ECG with a heart rate of 45 bpm and compare QTc by both formulas: Which one could label an athlete as abnormal? |
| Intensive care | Generate 5 minutes with `noise="clinic"` and measure Se and PPV of the R-peak detector against the true beat times. |

## After the notebook

Open the [interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lab.html) and take the self-assessment in the tab *Check yourself*. Then write the reflection in the tab *Reflect and export* and export the learning log. The weekly project of Week 1, described on the [week page](README.md), starts from the function `measure_ecg` of Section 5.

<div align="center">

---

**Prof. Dr. Utku Kose**<br>
Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University<br>
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists, Turkish Society of Cardiology. First version: November 2026. Content CC BY 4.0, code MIT.*

</div>